# Two-Tower Recommendation Model: Offline Evaluation & Diagnostics

This notebook demonstrates:
1. Loading trained user and item embedding representations from `data/processed/model/`
2. Constructing Approximate Nearest Neighbor (ANN) search indices (Exact vs. IVF)
3. Computing offline ranking metrics: **Recall@K**, **NDCG@K**, **MRR@K**, and **Hit Rate@K**
4. Benchmarking latency vs. recall retention trade-offs across multiple `nprobe` levels
5. Generating per-user diagnostic reports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Ensure workspace root is in python path
sys.path.append(os.path.abspath(".."))

from src.model.evaluate import ModelEvaluationPipeline
from src.retrieval.ann_search import ExactSearchIndex, IVFIndex

MODEL_DIR = "../data/processed/model"
print(f"Evaluating model artifacts from: {MODEL_DIR}")

## 1. Load Embeddings and Initialize Evaluation Pipeline

In [ ]:
pipeline = ModelEvaluationPipeline(model_dir=MODEL_DIR, k_values=(5, 10, 20))
pipeline.load_embeddings()
pipeline.build_exact_index()

print(f"User Embeddings  : {pipeline.user_embeddings.shape}")
print(f"Item Embeddings  : {pipeline.item_embeddings.shape}")
print(f"Index Built      : {pipeline.exact_index.num_items} items, dim={pipeline.exact_index.dim}")

## 2. Generate Synthetic Test Ground Truth & Compute Metrics

In [ ]:
# Construct synthetic ground truth from nearest-neighbor dot products
scores = np.matmul(pipeline.user_embeddings, pipeline.item_embeddings.T)
records = []
for u_idx, uid in enumerate(pipeline.user_ids):
    pos_idx = int(np.argmax(scores[u_idx]))
    records.append({
        "customer_id": str(uid),
        "article_id": str(pipeline.item_ids[pos_idx]),
    })

test_df = pd.DataFrame(records)
print(f"Constructed {len(test_df)} test interaction pairs.")

metrics = pipeline.evaluate(test_df)
for metric_name, val in sorted(metrics.items()):
    print(f"{metric_name:18s}: {val:.4f}")

## 3. ANN vs. Exact Latency/Recall Trade-Off Sweep

In [ ]:
sweep_df = pipeline.benchmark_ann_vs_exact(num_queries=min(50, len(pipeline.user_ids)), k=10)
display(sweep_df)

## 4. Per-User Diagnostic Report

In [ ]:
report_df = pipeline.generate_diagnostic_report(test_df, k_eval=10)
display(report_df.head(10))